# 21 · The encounter: from a prompt to a sentence in the Python monad (paper §13)
Code: `PtolemyDesktop/ptolemy_console.py` (`build_encounter_19d`, `ScaledMindEyeMonadAdapter.say`), `VAPMIP/19D_rotary_boxkite_monad.py` (`NineteenDMindsEye.create_context`, `RotaryBoxKite19D.process_input`), `VAPMIP/ptolemy_monad.py` (`infer_direction`).
This kernel runs with `PYTHONHASHSEED=0`; the last cells vary the seed.

In [1]:
from pn import *
L = Ledger("21 · the encounter")
with quiet():
    import ptolemy_console as pc
    ad = pc.ScaledMindEyeMonadAdapter()
print(ad.kind)
rounds = []
for rnd_i in (1, 2):
    rows = []
    for p in PROMPTS:
        t = time.perf_counter()
        with quiet(): reply, meta = ad.say(p)
        rows.append((p, reply, meta, time.perf_counter() - t))
    rounds.append(rows)
for (p, r1, m1, t1), (_, r2, m2, t2) in zip(*rounds):
    print(f"> {p}\n    {r1}\n    meta: { {k: m1[k] for k in m1 if k in ('via','direction','strut','verdict')} }   cold {t1*1000:.0f} ms, warm {t2*1000:.1f} ms")
L.check("within a process the second round returns the same replies as the first", all(a[1] == b[1] for a, b in zip(*rounds)))

scaled_mind_eye_boxkite_kernel_monad (real, wired 2026-09-26)


> finger pointer
    Pointer fingers.  [direction:decompose verdict:prime licensed:['SV', 'SVO']] [hub fiber: strut=6 fixed_point_weight=0.6400 is_zero_divisor=False]
    meta: {'via': 'scaled_mind_eye', 'direction': 'decompose', 'strut': 6, 'verdict': 'prime'}   cold 12840 ms, warm 18.8 ms
> what is a tree
    A ises tree.  [direction:situate verdict:prime licensed:['SVO']] [hub fiber: strut=5 fixed_point_weight=0.6400 is_zero_divisor=False]
    meta: {'via': 'scaled_mind_eye', 'direction': 'situate', 'strut': 5, 'verdict': 'prime'}   cold 354 ms, warm 55.6 ms
> neutron generation in different reactor types
    (19D: direction='compare' has no speech_act mapping yet — reporting the real finding rather than forcing one: leaves=['neutron', 'generation', 'in', 'different', 'reactor'])
    meta: {'via': '19D_unmapped', 'direction': 'compare'}   cold 12 ms, warm 0.3 ms
> good morning
    (19D: nothing resolved — no WordNet verb found, 2 content word(s) resolved. NULL, not fabricated — no s

True

## NULL first: nothing is fabricated when nothing resolves

In [2]:
nulls = [(p, r) for p, r, m, t in rounds[1] if r.startswith("(19D:")]
for p, r in nulls: print(f"> {p}\n    {r}")
kinds = {m.get("via") for p, r, m, t in rounds[1] if r.startswith("(19D:")}; print("non-sentence replies by kind:", sorted(kinds))
L.check("some test prompts end in an explicit non-sentence reply (no verb: NULL; no dominant signal: observe; direction without a speech act: unmapped), none fabricated", len(nulls) >= 1, f"{len(nulls)} of {len(PROMPTS)}: {sorted(kinds)}")
L.check("each such reply says why and carries no generated sentence (no hub-fiber line)", all("hub fiber" not in r for p, r in nulls))
answered = [(p, r, m) for p, r, m, t in rounds[1] if m.get("via") == "scaled_mind_eye"]
L.check("a resolved prompt reports its direction, its strut and the fixed-point weight of the throw", all("hub fiber" in r and "fixed_point_weight" in r for p, r, m in answered), f"{len(answered)} resolved")

> neutron generation in different reactor types
    (19D: direction='compare' has no speech_act mapping yet — reporting the real finding rather than forcing one: leaves=['neutron', 'generation', 'in', 'different', 'reactor'])
> good morning
    (19D: nothing resolved — no WordNet verb found, 2 content word(s) resolved. NULL, not fabricated — no sentence generated.)
> the first reactor was built after neutrons were discovered
    (19D: direction='compare' has no speech_act mapping yet — reporting the real finding rather than forcing one: leaves=['first', 'reactor', 'built', 'after', 'neutrons', 'were', 'discovered'])
non-sentence replies by kind: ['19D_null', '19D_unmapped']


True

## Across hash seeds (five fresh processes)

In [3]:
CHILD = r'''
import sys, json, io, contextlib
sys.path.insert(0, %r); sys.path.insert(0, %r)
q = io.StringIO()
with contextlib.redirect_stdout(q), contextlib.redirect_stderr(q):
    import ptolemy_console as pc
    ad = pc.ScaledMindEyeMonadAdapter()
    out = [ad.say(p)[0] for p in %r]
print(json.dumps(out))
''' % (VAPMIP, os.path.join(PLACE, "PtolemyDesktop"), PROMPTS)
seeds, per = [0, 1, 2, 3, 4], {p: [] for p in PROMPTS}
for sd in seeds:
    r = subprocess.run([sys.executable, "-c", CHILD], capture_output=True, text=True, env=dict(os.environ, PYTHONHASHSEED=str(sd)), timeout=600)
    for p, rep in zip(PROMPTS, json.loads(r.stdout.strip().splitlines()[-1])): per[p].append(rep)
for p in PROMPTS: print(f"{len(set(per[p]))} distinct | {p}")
nvar = sum(1 for p in PROMPTS if len(set(per[p])) > 1)
L.check("replies are fixed for a given seed (seed 0 here equals seed 0 in a fresh process)", all(per[p][0] == rounds[1][i][1] for i, p in enumerate(PROMPTS)))
L.check("replies do not depend on the hash seed (a measured disagreement is kept if it does)", nvar == 0, f"{nvar} of {len(PROMPTS)} prompts give more than one reply across seeds 0–4")

1 distinct | finger pointer
1 distinct | what is a tree
1 distinct | neutron generation in different reactor types
1 distinct | good morning
1 distinct | can you play the piano?
1 distinct | the first reactor was built after neutrons were discovered


True

## Ledger

In [4]:
_n = L.show()

LEDGER — 21 · the encounter
  [PASS ] within a process the second round returns the same replies as the first
  [PASS ] some test prompts end in an explicit non-sentence reply (no verb: NULL; no dominant signal: observe; direction without a speech act: unmapped), none fabricated
          3 of 6: ['19D_null', '19D_unmapped']
  [PASS ] each such reply says why and carries no generated sentence (no hub-fiber line)
  [PASS ] a resolved prompt reports its direction, its strut and the fixed-point weight of the throw
          3 resolved
  [PASS ] replies are fixed for a given seed (seed 0 here equals seed 0 in a fresh process)
  [PASS ] replies do not depend on the hash seed (a measured disagreement is kept if it does)
          0 of 6 prompts give more than one reply across seeds 0–4
  -> PASS 6 · FAIL 0 · PROSE 0
